## **SNAP Jupyter demo notebook**
**Using Read, BandMaths, BandMerge, Subset and Write operators...** 

In summary, this workflow contains:

- Read a subset of an S3 OLCI data product using the SNAP 'Read' operator
- Compute a simple NDVI using the SNAP 'BandMaths' operator
- Merge the BandMaths target product (NDVI) with the source product using the SNAP 'BandMerge' operator
- Extract three bands from the 'Merge' product (two input radiances and the NDVI) using the SNAP 'Subset' operator 
- Write the result to a NetCDF file using the SNAP 'Write' operator
- Combine all steps (nodes) into a SNAP GPF xml graph, and run the graph using SNAPISTA

Complexity: intermediate

##### ***Some information on the Python environment:***

In [74]:
import os
import sys
print("Python version: " + sys.version)

import sysconfig
print("Location of esa_snappy package: " + sysconfig.get_paths()['purelib'] + os.sep + "esa_snappy")

Python version: 3.13.11 | packaged by Anaconda, Inc. | (main, Dec 10 2025, 21:21:58) [MSC v.1929 64 bit (AMD64)]
Location of esa_snappy package: E:\olaf\miniconda_py313\Lib\site-packages\esa_snappy


##### ***Import esa_snappy and snapista:***

In [75]:
import esa_snappy
import snapista

from snapista import Graph
from snapista import Operator
from snapista import read_file
from snapista import TargetBand
from snapista import TargetBandDescriptors

##### ***Set up a graph:***

In [76]:
try:
    g = Graph()
except Exception as ex:
    print("Cannot set up Snapista Graph():", ex)

##### ***Add a 'Read' operator to the graph for reading example input:***

In [77]:
input_dir = os.getcwd() + os.sep + 'data'
input_filename = input_dir + os.sep + 'subset_0_of_S3A_OL_1_ERR____20160701T092853_20160701T093053_20170919T102725_0119_006_036______MR1_R_NT_002.nc'
g.add_node(operator=Operator("Read", 
                             file=input_filename), 
           node_id="Read")

##### ***Set up a 'BandMaths' operator to compute a simple NDVI. Set up a target band for NDVI result:***

In [78]:
band_maths = Operator('BandMaths')

ndvi = TargetBand(name='ndvi', 
                  expression='(Oa08_radiance - Oa17_radiance)/(Oa08_radiance + Oa17_radiance)')                     
band_maths.targetBandDescriptors = TargetBandDescriptors([ndvi])

##### ***Add 'BandMaths' operator to the graph:***

In [79]:
g.add_node(operator=band_maths, 
           node_id="BandMaths" ,
           source='Read')

targetBandDescriptors <snapista.target_band_descriptors.TargetBandDescriptors object at 0x000001CD30CBDFD0>
Instance TargetBandDescriptors: True
Instance Aggregators: False
Instance BinningOutputBands: False
Instance BinningVariables: False
Instance str: False


##### *** Set up a 'BandMerge' operator to merge input with NDVI result:***

In [80]:
merge = Operator("BandMerge", 
                 source=["Read", "BandMaths"])

##### ***Add 'Merge' operator to the graph:***

In [81]:
g.add_node(operator=merge, 
           node_id="Merge", 
           source=["Read", "BandMaths"])

##### ***Set up a 'Subset' operator to extract just three bands:***

In [82]:
subset = Operator('Subset', 
                  bandNames="Oa08_radiance,Oa17_radiance,ndvi")

##### ***Add 'Subset' operator to the graph:***

In [83]:
g.add_node(operator=subset, 
           node_id="Subset",
           source='Merge')

##### ***Set up a 'Write' operator to write final target product with the three bands:***

In [84]:
write = Operator("Write")

##### ***Add 'Write' operator to the graph. Define a NetCDF output file:***

In [85]:
result_filename = os.getcwd() + os.sep + 'results' + os.sep + 'snap_nb_olci_bandmaths_merge_subset_result.nc'
g.add_node(operator=Operator("Write", 
                             file=result_filename, 
                             formatName='NetCDF4-BEAM'), 
           node_id="Write", 
           source='Subset')

##### ***View graph in XML representation:***

In [86]:
g.view()

<graph>
  <version>1.0</version>
  <node id="Read">
    <operator>Read</operator>
    <sources/>
    <parameters class="com.bc.ceres.binding.dom.XppDomElement">
      <bandNames/>
      <copyMetadata>true</copyMetadata>
      <file>E:\olaf\bc\snap\snap_md\notebooks\new\data\subset_0_of_S3A_OL_1_ERR____20160701T092853_20160701T093053_20170919T102725_0119_006_036______MR1_R_NT_002.nc</file>
      <formatName/>
      <geometryRegion/>
      <maskNames/>
      <pixelRegion/>
      <polygonRegion/>
      <useAdvancedOptions>false</useAdvancedOptions>
      <vectorFile/>
    </parameters>
  </node>
  <node id="BandMaths">
    <operator>BandMaths</operator>
    <sources>
      <source refid="Read"/>
    </sources>
    <parameters class="com.bc.ceres.binding.dom.XppDomElement">
      <targetBands>
        <targetBand>
          <name>ndvi</name>
          <expression>(Oa08_radiance - Oa17_radiance)/(Oa08_radiance + Oa17_radiance)</expression>
          <type>float32</type>
          <descripti

##### ***Save graph as XML file:***

In [87]:
saved_graph_name = os.getcwd() + os.sep + 'graphs' + os.sep + 'snap_nb_olci_bandmaths_merge_subset.xml'
g.save_graph(saved_graph_name)

##### ***Run the graph:***

In [88]:
g.run()

Processing the graph, this may take a while. Please wait...
standard output of subprocess: 
Executing processing graph
15%30%45%60%...70%..90% done.
standard error of subprocess: 
INFO: org.esa.snap.core.gpf.operators.tooladapter.ToolAdapterIO: Initializing external tool adapters
INFO: org.esa.snap.core.util.EngineVersionCheckActivator: Please check regularly for new updates for the best SNAP experience.
[main] INFO serverStartup - Nc4Iosp: NetCDF-4 C library loaded (jna_path='C:\Users\olafd\.snap\auxdata\netcdf_natives\13.0.0\amd64', libname='netcdf').
[main] INFO serverStartup - NetcdfLoader: set log level: old=0 new=0
[main] INFO serverStartup - Nc4Iosp: set log level: old=0 new=0
Processing finished successfully.


0